In [ ]:
# Standard library imports
import sys
import time

# Third-party imports
import numpy as np
import torch
from matplotlib import pyplot as plt
from scipy.fftpack import idct
from torch.utils.data import ConcatDataset, random_split, TensorDataset
from torchdiffeq import odeint
from tqdm import tqdm
from itertools import product
import sourcedefender

# System path modifications
sys.path.append("./")
sys.path.append("../")
sys.path.append("../../")
sys.path.append("../../../")
sys.path.append("/")

# Local/custom module imports
from lib.batchJacobian import batchJacobian_PDE
from lib.helper import *
from operators import DifferentialOperators
from PPT_solver import solve_poisson


In [2]:
# Set up computation device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize MHPI (assumed to be a setup or initialization function)
MHPI()

# Log the selected device
print(f'Using device: {device}')

Presented by 
               __  __ _   _ ____ ___    ____                       
             |  \/  | | | |  _ \_ _|  / ___|_ __ ___  _   _ _ __  
             | |\/| | |_| | |_) | |  | |  _| '__/ _ \| | | | '_ \ 
             | |  | |  _  |  __/| |  | |_| | | | (_) | |_| | |_) |
             |_|  |_|_| |_|_|  |___|  \____|_|  \___/ \__,_| .__/ 
                                                           |_|    

Using device: cuda


In [ ]:
def GRF(alpha: float, tau: float, s: int, A: float, B: float) -> np.ndarray:
    """
    Generate a single Gaussian Random Field (GRF) using the Karhunen-Loève expansion.

    Args:
        alpha (float): Smoothness parameter controlling the field regularity.
        tau (float): Length scale parameter of the covariance operator.
        s (int): Grid size (s x s) for the spatial field.
        A (float): Lower bound for scaling the field.
        B (float): Upper bound for scaling the field.

    Returns:
        np.ndarray: Scaled GRF of shape (s, s) with values in [A, B].
    """
    # Generate random variables for KL expansion
    random_variables = np.random.normal(loc=0, scale=1, size=(s, s))

    # Create meshgrid for eigenvalue computation
    k1, k2 = np.meshgrid(range(s), range(s), indexing='ij')

    # Compute square root of eigenvalues for the covariance operator
    eigenvalues = tau ** (alpha - 1) * (np.pi**2 * (k1**2 + k2**2) + tau**2) ** (-alpha / 2)

    # Construct KL coefficients
    kl_coefficients = s * eigenvalues * random_variables
    kl_coefficients[0, 0] = 0  # Zero out the (0,0) component

    # Apply inverse discrete cosine transform to obtain the spatial field
    spatial_field = idct(idct(kl_coefficients, norm='ortho', axis=0), norm='ortho', axis=1)

    # Normalize to zero mean and unit variance
    normalized_field = (spatial_field - np.mean(spatial_field)) / np.std(spatial_field)

    # Scale to the interval [A, B]
    field_min, field_max = normalized_field.min(), normalized_field.max()
    scaled_field = A + (normalized_field - field_min) / (field_max - field_min) * (B - A)

    return scaled_field

def batch_grf(alpha: float, tau: float, s: int, A: float, B: float, batch_size: int) -> torch.Tensor:
    """
    Generate a batch of Gaussian Random Fields as a PyTorch tensor.

    Args:
        alpha (float): Smoothness parameter controlling the field regularity.
        tau (float): Length scale parameter of the covariance operator.
        s (int): Grid size (s x s) for each spatial field.
        A (float): Lower bound for scaling the fields.
        B (float): Upper bound for scaling the fields.
        batch_size (int): Number of GRF samples to generate.

    Returns:
        torch.Tensor: Batch of GRFs with shape (batch_size, s, s).
    """
    # Generate batch of GRF samples
    batch_data = [GRF(alpha, tau, s, A, B) for _ in range(batch_size)]

    # Convert to PyTorch tensor
    batch_tensor = torch.tensor(batch_data, dtype=torch.float32)

    return batch_tensor

In [ ]:
def forcing(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    Compute a forcing term based on input coordinates x and y.

    The forcing term is defined as sin(2π((x*y) + x)) + cos(2π((x*y) + y)).

    Args:
        x (torch.Tensor): Input tensor representing x-coordinates.
        y (torch.Tensor): Input tensor representing y-coordinates.

    Returns:
        torch.Tensor: Forcing term computed as a combination of sine and cosine functions.
    """
    # Compute the forcing term: sin(2π((x*y) + x)) + cos(2π((x*y) + y))
    xy_product = x * y
    sine_term = torch.sin(2 * np.pi * (xy_product + x))
    cosine_term = torch.cos(2 * np.pi * (xy_product + y))
    return sine_term + cosine_term

In [ ]:
def rhs(
    t: float,
    w_flat: torch.Tensor,
    operators: 'DifferentialOperators',
    F: torch.Tensor,
    nu: float,
    N: int,
    device: torch.device,
    dx: float,
    dy: float
) -> torch.Tensor:
    """
    Compute the right-hand side of a vorticity-based PDE using differential operators.

    This function calculates the time derivative of the vorticity field (dw/dt) for a PDE,
    typically used in an ODE solver. It involves solving a Poisson equation, computing gradients,
    and applying a forcing term.

    Args:
        t (float): Current time (unused in computation but required by ODE solver convention).
        w_flat (torch.Tensor): Flattened vorticity tensor of shape (batch_size, N*N).
        operators (DifferentialOperators): Instance of DifferentialOperators for computing gradients and Laplacian.
        F (torch.Tensor): Forcing term tensor, typically of shape (N, N).
        nu (float): Viscosity coefficient.
        N (int): Grid size (N x N) for the spatial field.
        device (torch.device): Device (CPU/GPU) for tensor computations.
        dx (float): Grid spacing in the x-direction.
        dy (float): Grid spacing in the y-direction.

    Returns:
        torch.Tensor: Flattened time derivative of vorticity (dw/dt) of shape (batch_size, N*N).
    """
    # Reshape flattened vorticity into [batch_size, N, N]
    w = w_flat.view(-1, N, N)

    # Solve Poisson equation to obtain stream function psi
    psi = solve_poisson(w, device, N, dx, dy)

    # Compute gradients of stream function and vorticity
    dpsi_dx, dpsi_dy = operators.grad(psi)
    dw_dx, dw_dy = operators.grad(w)

    # Compute Jacobian: dpsi_dy * dw_dx - dpsi_dx * dw_dy
    jacobian = dpsi_dy * dw_dx - dpsi_dx * dw_dy

    # Compute Laplacian of vorticity
    lap_w = operators.laplacian(w)

    # Expand forcing term to match batch dimensions
    F_expanded = F.unsqueeze(0).expand_as(w)

    # Compute right-hand side: nu * Laplacian(w) - Jacobian + F
    dwdt = nu * lap_w - jacobian + F_expanded

    # Flatten output for ODE solver compatibility
    return dwdt.view(-1, N * N)

In [ ]:
def solver(
    w0_batch: torch.Tensor,
    t_span: torch.Tensor,
    operators: 'DifferentialOperators',
    F: torch.Tensor,
    nu: float,
    N: int,
    device: torch.device,
    dx: float,
    dy: float
) -> torch.Tensor:
    """
    Solve a vorticity-based PDE using the provided initial conditions and parameters.

    This function uses an ODE solver to compute the evolution of the vorticity field
    over a specified time span, leveraging the right-hand side function `rhs`.

    Args:
        w0_batch (torch.Tensor): Initial vorticity conditions, shape (batch_size, N, N).
        t_span (torch.Tensor): Array of time points for solving the PDE, shape (num_times,).
        operators (DifferentialOperators): Instance of DifferentialOperators for computing derivatives.
        F (torch.Tensor): Forcing term, shape (1, N, N) or (N, N).
        nu (float): Viscosity coefficient.
        N (int): Grid size (N x N) for the spatial field.
        device (torch.device): Device (CPU/GPU) for tensor computations.
        dx (float): Grid spacing in the x-direction.
        dy (float): Grid spacing in the y-direction.

    Returns:
        torch.Tensor: Solution of the PDE, shape (num_times, batch_size, N*N).
    """
    # Flatten initial conditions to shape (batch_size, N*N) for ODE solver
    w0_flat = w0_batch.view(-1, N * N)

    # Define the right-hand side function with extra arguments for odeint
    rhs_func = lambda t, w_flat: rhs(t, w_flat, operators, F, nu, N, device, dx, dy)

    # Solve the PDE using odeint
    w_flat_sol = odeint(
        func=rhs_func,
        y0=w0_flat,
        t=t_span,
        method='implicit_adams'
    )

    return w_flat_sol

In [ ]:
def create_and_save_dataset(
    device: torch.device,
    initial_conditions: torch.Tensor,
    nu: float = 0.001,
    dataset_segment_size: int = 10,
    N: int = 64,
    t0: float = 0.0,
    t_end: float = 3.0,
    steps: int = 150
) -> TensorDataset:
    """
    Create a dataset by solving a PDE for batches of initial conditions and computing Jacobians.

    This function generates a dataset containing initial conditions, final PDE solutions, and their
    Jacobians with respect to the initial conditions, processed in segments to manage memory.

    Args:
        device (torch.device): Device (CPU/GPU) for tensor computations.
        initial_conditions (torch.Tensor): Batch of initial vorticity conditions, shape (total_samples, N, N).
        nu (float, optional): Viscosity coefficient. Defaults to 0.001.
        dataset_segment_size (int, optional): Number of samples per batch segment. Defaults to 10.
        N (int, optional): Grid size (N x N) for the spatial field. Defaults to 64.
        t0 (float, optional): Start time for PDE integration. Defaults to 0.0.
        t_end (float, optional): End time for PDE integration. Defaults to 3.0.
        steps (int, optional): Number of time steps for PDE integration. Defaults to 150.

    Returns:
        TensorDataset: Dataset containing initial conditions, final solutions, and Jacobians.
    """
    # Initialize spatial and temporal grids
    t_tensor = torch.linspace(t0, t_end, steps, dtype=torch.float32, device=device)
    x = torch.linspace(0, 1, N, dtype=torch.float32, device=device)
    y = torch.linspace(0, 1, N, dtype=torch.float32, device=device)
    dx = x[1] - x[0]

    # Initialize differential operators
    operators = DifferentialOperators(N, dx, device)

    # Compute forcing term on spatial grid
    X, Y = torch.meshgrid(x, y, indexing='ij')
    F = forcing(X, Y)

    # Initialize lists to store dataset components
    batch_initial_conditions_list = []
    u_list = []
    jac_list = []

    # Process initial conditions in segments
    num_segments = len(initial_conditions) // dataset_segment_size
    for i in range(num_segments):
        t_start = time.time()

        # Extract batch of initial conditions
        start_idx = i * dataset_segment_size
        end_idx = (i + 1) * dataset_segment_size
        batch_initial_conditions = initial_conditions[start_idx:end_idx].to(device)
        batch_initial_conditions.requires_grad_(True)
        batch_size = batch_initial_conditions.shape[0]

        # Solve PDE for the batch
        u = solver(
            batch_initial_conditions, t_tensor, operators, F, nu, N, device, dx, dy=dx
        ).permute(1, -1, 0).reshape(batch_size, N, N, steps)

        # Extract and downsample the solution at the final time step
        N_reduce = 2
        solution_last = u[:, ::N_reduce, ::N_reduce, -1]

        # Compute Jacobian of the final solution with respect to initial conditions
        jac = batchJacobian_PDE(solution_last, batch_initial_conditions)
        jac_upscaled = upscale_tensor(jac=jac, N=N_reduce)

        # Store detached tensors
        batch_initial_conditions_list.append(batch_initial_conditions.detach())
        u_list.append(u[:, :, :, -1].detach())
        jac_list.append(jac_upscaled.detach())

        print(f'Prepared segment {i+1}/{num_segments}, time: {time.time() - t_start:.3f} s')

    # Concatenate results into final tensors
    batch_initial_conditions_final = torch.cat(batch_initial_conditions_list, dim=0)
    u_final = torch.cat(u_list, dim=0)
    jac_final = torch.cat(jac_list, dim=0)

    # Create and return dataset
    dataset = TensorDataset(batch_initial_conditions_final, u_final, jac_final)
    return dataset

In [ ]:
# Configuration
case = 'nse_multi_ic_J_ic'
PATH_SAVE_DATASET = '/scratch/amb10399/DATA/NSE'

# Ensure the save directory exists
ensure_directory(PATH_SAVE_DATASET)

# Parameters for dataset generation
n_samples = 10
N = 50
alpha = 3.0
tau = 0.5
A = -1.0
B = 1.0

# Generate initial conditions using GRF
initial_conditions = batch_grf(
    alpha=alpha,
    tau=tau,
    s=N,
    A=A,
    B=B,
    batch_size=n_samples
).to(device)

# Generate and save dataset
t_start = time.time()
dataset = create_and_save_dataset(
    device=device,
    initial_conditions=initial_conditions,
    nu=0.001,
    dataset_segment_size=10,
    N=N,
    t0=0.0,
    t_end=3.0,
    steps=100
)

# Save dataset to file
torch.save(dataset, f'{PATH_SAVE_DATASET}/{case}_main_dataset.pt')

# Log total execution time
print(f'Overall CPU time: {time.time() - t_start:.3f} seconds')

In [ ]:
dataset.tensors[0].shape, dataset.tensors[1].shape, dataset.tensors[2].shape  
# Should be 
# a: [nb, nx, ny] 
# u: [nb, nx, ny] 
# jac (u) w.r.t (a): [nb, nx, ny, nx, ny] 

In [ ]:
import matplotlib.pyplot as plt
import torch

# Extract dataset tensors
a, u, jac = dataset.tensors  # Shapes: [nb, nx, ny], [nb, nx, ny], [nb, nx, ny, nx, ny]

# Select two samples for plotting
sample_indices = [0, 1]  # Plot first two samples
nb, nx, ny = a.shape

# Create a 2-row, 3-column subplot grid
fig, axes = plt.subplots(2, 3, figsize=(12, 8), sharex=True, sharey=True)

# Titles for columns
titles = ['Initial Condition (a)', 'Final Solution (u)', 'Jacobian (jac[:, :, 25, 25])']

# Plot two samples
for i, idx in enumerate(sample_indices):
    # Move tensors to CPU and convert to numpy for plotting
    a_sample = a[idx].cpu().numpy()  # Shape: [nx, ny]
    u_sample = u[idx].cpu().numpy()  # Shape: [nx, ny]
    jac_sample = jac[idx, :, :, 25, 25].cpu().numpy()  # Shape: [nx, ny]

    # Plot initial condition (a)
    im0 = axes[i, 0].imshow(a_sample, cmap='viridis', origin='lower')
    axes[i, 0].set_title(f'{titles[0]} (Sample {idx+1})')
    fig.colorbar(im0, ax=axes[i, 0], fraction=0.046, pad=0.04)

    # Plot final solution (u)
    im1 = axes[i, 1].imshow(u_sample, cmap='viridis', origin='lower')
    axes[i, 1].set_title(f'{titles[1]} (Sample {idx+1})')
    fig.colorbar(im1, ax=axes[i, 1], fraction=0.046, pad=0.04)

    # Plot Jacobian slice (jac[:, :, 0, 0])
    im2 = axes[i, 2].imshow(jac_sample, cmap='viridis', origin='lower')
    axes[i, 2].set_title(f'{titles[2]} (Sample {idx+1})')
    fig.colorbar(im2, ax=axes[i, 2], fraction=0.046, pad=0.04)

# Set labels and adjust layout
for ax in axes.flat:
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.tight_layout()
plt.show()